In [1]:
from __future__ import annotations

import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torchvision.models as tvm
from mlflow.models import infer_signature

In [2]:
# --- edit these ---
CHECKPOINT_PATH: str = "/home/david/coin/weights/emp_model_mobilenet_baseline.pth"      # your fine-tuned state dict
TRACKING_URI: str = "http://127.0.0.1:5000"
INPUT_SIZE: int = 224                          # MUST match your training/inference transform

# --- stable ---
MODEL_NAME: str = "coin-classifier"
EXPERIMENT_NAME: str = "coin-classifier"
NUM_CLASSES: int = 51
ARCH: str = "mobilenet_v3_large"

In [3]:

def build_model(num_classes: int) -> nn.Module:
    model = tvm.mobilenet_v3_large(weights=None)  # weights=None: the state dict overwrites everything anyway
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    return model

In [4]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

model = build_model(NUM_CLASSES)
state_dict = torch.load(CHECKPOINT_PATH, map_location="cpu")
model.load_state_dict(state_dict)  # strict=True by default: fails loudly on any key mismatch
model.eval()

# signature + input example so the logged model is self-describing for serving
example_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
with torch.no_grad():
    example_output = model(example_input)
signature = infer_signature(example_input.numpy(), example_output.numpy())

with mlflow.start_run(run_name="seed-champion") as run:
    mlflow.log_params(
        {
            "arch": ARCH,
            "num_classes": NUM_CLASSES,
            "input_size": INPUT_SIZE,
            "source_checkpoint": CHECKPOINT_PATH,
        }
    )
    model_info = mlflow.pytorch.log_model(
    pytorch_model=model,
    name="model",
    signature=signature,
    input_example=example_input.numpy(),
    serialization_format="pickle",   # <-- add this; pt2 needs torch>=2.4, you're older
)
    run_id = run.info.run_id

mv = mlflow.register_model(model_info.model_uri, MODEL_NAME)
client = mlflow.MlflowClient()
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)

champ = client.get_model_version_by_alias(MODEL_NAME, "champion")
print(f"run_id={run_id}")
print(f"{MODEL_NAME} v{champ.version} registered, alias @champion set")


2026/08/20 11:25:41 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
Successfully registered model 'coin-classifier'.
2026/08/20 11:25:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: coin-classifier, version 1


🏃 View run seed-champion at: http://127.0.0.1:5000/#/experiments/2/runs/f731f0d65bb043cd98eeabccbd62a8ed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


Created version '1' of model 'coin-classifier'.


run_id=f731f0d65bb043cd98eeabccbd62a8ed
coin-classifier v1 registered, alias @champion set
